# 2장 2강: 신경망 최적화 — 실습문제


## 실습 목표

- 역전파와 Optimizer의 역할을 구분합니다.
- 기본 SGD의 업데이트 식을 사용하고 Learning Rate에 따른 차이를 확인합니다.
- Adam이 현재 기울기와 이전 기울기 기록을 이용하는 과정을 따라갑니다.
- 같은 초기값·샘플·학습률·업데이트 횟수로 두 방식의 계산 결과를 비교합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용하며 PyTorch 설치는 필요하지 않습니다.
- 이전 강의와 같은 **`concrete_data.csv`**를 사용합니다. 노트북과 CSV를 같은 폴더에 두고 그 폴더를 VS Code에서 열어 실행하세요.
- pandas와 NumPy가 설치된 Python 환경을 커널로 선택합니다. 준비 셀부터 위에서 아래로 실행하세요.
- 원본은 1030개 행, 입력 후보 특성 8개와 실제 압축강도 열 1개로 구성됩니다.
- 이전 강의와 같이 **시멘트량 / 100**을 입력 하나로 사용합니다. 실제 정답 압축강도는 원래 값 그대로입니다.
- 한 특성과 일부 샘플만 사용하는 것은 계산 원리를 익히기 위해서입니다. 나머지 특성이 필요 없다는 결론은 아닙니다.
- 실습은 첫 샘플, 과제는 세 번째 샘플을 사용합니다.

## 이번 강의의 범위

제공된 **2장 2강 목차: SGD와 Adam, Optimizer의 동작 원리와 차이, Learning Rate와 파라미터 업데이트**를 기준으로 구성했습니다. 강의 본문은 별도로 제공되지 않았으므로 아래 숫자와 모델은 개념 설명용 예시입니다.

역전파는 2장 1강에서 사용한 작은 모델을 제공 함수로 묶어 사용합니다. 학생이 미분 식이나 함수를 새로 작성할 필요는 없습니다. 이번에는 **구한 기울기로 파라미터를 어떻게 바꾸는지**에 집중합니다.

SGD는 모멘텀 등을 추가하지 않은 기본형을 사용하고, Adam은 m·v의 누적과 초기값 보정을 포함합니다. 증명이나 공식 암기는 요구하지 않습니다. 학습률 스케줄러, 다른 Optimizer, 정규화, 전체 데이터 학습·검증, 성능 튜닝은 다루지 않습니다.

## 실습 구성

| 구분 | 문제 | 할 일 | 강의 내용 |
|---|---|---|---|
| 실습 1 | 1-1 | SGD에서 학습률만 바꾼 한 번의 업데이트 비교 | Learning Rate와 파라미터 업데이트 |
| 실습 2 | 2-1 | Adam의 첫 업데이트와 m·v 확인 | Adam의 동작 원리 |
| 실습 2 | 2-2 | 두 번째 업데이트에서 기록 유지, SGD와 비교 | Optimizer의 차이 |
| 기초 과제 | 3-1 | 세 번째 샘플에서 SGD·Adam 한 번씩 적용 | 실습의 기본 계산 복습 |

**실습 2개 + 기초 과제 1개**입니다. 실습 2는 Adam의 첫 단계와 다음 단계를 나눴습니다. 과제는 같은 식을 다른 샘플에 적용하는 수준입니다.

## 진행 안내

1. 실습 준비 코드는 제공 코드입니다. 그대로 실행한 뒤 문제를 시작하세요.
2. 파라미터 배열과 기울기 배열의 순서는 항상 **[w1, b1, w2, b2]**입니다.
3. 파라미터를 바꿨다면 다음 업데이트 전에 그 파라미터에서 기울기를 다시 계산합니다.
4. 같은 학습을 이어갈 때 Adam의 m·v·t를 유지합니다. 독립적인 새 실험을 시작할 때만 초기화합니다.
5. 출력할 때만 반올림하고, 계산에는 반올림 전 값을 사용합니다.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습 |
|---|---|---|
| Optimizer | 기울기를 받아 파라미터를 수정하는 규칙 | SGD, Adam |
| 역전파 | 손실을 기준으로 파라미터별 기울기를 구함 | 제공 함수 안에서 계산 |
| 파라미터 theta | 모델의 가중치·편향을 모은 값 | [w1, b1, w2, b2] |
| 기울기 g | 각 파라미터를 조금 바꿀 때 손실이 변하는 방향·정도 | theta와 같은 순서의 숫자 4개 |
| 업데이트 | 기존 파라미터를 새 값으로 바꿈 | theta_new 계산 |
| Learning Rate, lr | 수정 규모를 조절하는 기본값 | 0.0001 등 |
| SGD | 샘플 또는 작은 묶음으로 기울기를 구해 업데이트하는 방식 | 이번에는 샘플 하나의 기본 업데이트 식 |
| Adam | 기울기와 제곱 기울기의 기록으로 수정량을 조절하는 방식 | m·v를 유지하며 계산 |
| m | 기울기의 방향·크기를 누적한 이동평균 | 이전 m과 현재 g를 섞음 |
| v | 기울기를 제곱한 값의 이동평균 | 큰 기울기가 얼마나 있었는지 반영 |
| beta1, beta2 | 이전 기록과 새 값을 섞는 비율 | 0.9, 0.999로 고정 |
| t | 지금까지 진행한 업데이트의 순번 | 첫 번째 1, 두 번째 2 |
| epsilon, eps | 분모가 0이 되는 것을 피하는 작은 양수 | 1e-8 = 0.00000001 |

여기서 v는 **제곱 기울기의 평균**이며, 평균을 뺀 뒤 제곱하는 통계적 분산과 같은 계산은 아닙니다. Adam의 ‘초기값 편향 보정’은 모델의 편향 b1·b2를 조정한다는 뜻과도 다릅니다.


### 앞 강의에서 이번 강의로 연결하기

2장 1강에서는 역전파로 기울기를 구하고 그 값을 이용해 파라미터를 한 번 바꿨습니다. 이번에는 **어떤 규칙으로 바꾸느냐**를 비교합니다. Optimizer는 ‘옵티마이저’라고 읽고, 여기서는 손실을 줄이도록 모델의 가중치·편향을 수정하는 방법을 뜻합니다.

| 구분 | 하는 일 | 이번 실습의 위치 |
|---|---|---|
| 순전파 | 현재 파라미터로 출력 계산 | 제공 함수 내부 |
| 손실 계산 | 출력과 실제 정답의 차이를 숫자로 계산 | 제공 함수 내부 |
| 역전파 | 각 파라미터가 조금 변할 때 손실이 어떻게 변하는지 계산 | 제공 함수 내부 |
| Optimizer의 업데이트 | 그 기울기를 이용해 새 파라미터 계산 | 학생이 작성하는 SGD·Adam 식 |

Optimizer는 실제 정답을 가중치에 그대로 넣거나 손실 숫자 자체를 모든 파라미터에서 빼는 것이 아닙니다. **파라미터마다 대응하는 기울기**를 사용합니다. 이번에는 기울기를 제공 함수로 구하고, 업데이트 식에 집중합니다.

### 기호와 배열 읽기

| 표시 | 쉽게 읽기 | 구분할 것 |
|---|---|---|
| $	heta$, `theta` | ‘세타’, 가중치·편향을 모아 부르는 이름 | 별도의 다섯 번째 파라미터가 아님 |
| g | 기울기 배열 | 파라미터 배열과 같은 순서로 대응 |
| t, t-1 | 이번 업데이트 순번, 직전 순번 | 데이터 행 번호나 층 번호가 아님 |
| $m_t$, $v_t$ | t번째 단계의 m 기록, v 기록 | 아래 t는 제곱이 아니라 단계 표시 |
| $hat m$, $hat v$ | 초기값 영향을 보정한 m, v | 위의 표시는 ‘보정한 값’을 구분하는 표시 |
| $sqrt{a}$ | 제곱해서 a가 되는 0 이상의 수 | 예: √9 = 3, 코드에서는 `np.sqrt(a)` |
| $epsilon$, `eps` | ‘엡실론’, 분모에 더할 작은 양수 | 이번 값은 0.00000001 |

`theta0 = [0.5, 0.1, 20, 20]`의 0.5는 w1, 0.1은 b1, 뒤의 두 20은 각각 w2와 b2입니다. 예를 들어 `theta0[0]`은 첫 위치의 w1입니다. Python의 위치 번호는 0부터 시작합니다. 같은 위치의 기울기를 연결해야 올바른 값을 수정할 수 있습니다.

### 기초 수준에서 공부하는 순서

먼저 SGD의 **기존 값 − 학습률 × 기울기**를 이해하세요. Adam은 **기록 갱신 → 초기값 보정 → 뺄 양 계산 → 새 파라미터**의 네 단계로 나누어 읽습니다. 처음부터 네 파라미터 전체의 긴 소수를 손으로 계산할 필요는 없습니다. w1 한 개를 따라가고 나머지는 같은 규칙으로 코드가 계산한 결과를 확인하면 됩니다.


## 실습 준비 — 두 코드 셀을 그대로 실행하세요

원본 열 이름의 끝 공백을 제거하고 데이터를 읽습니다. 모델은 **입력 1개 → ReLU 은닉 뉴런 1개 → 출력 뉴런 1개**이며, 초기 파라미터는 다음과 같습니다.

| 배열 위치 | 파라미터 | 초기값 |
|---:|---|---:|
| 0 | w1 | 0.5 |
| 1 | b1 | 0.1 |
| 2 | w2 | 20.0 |
| 3 | b2 | 20.0 |

$$z1=w1x+b1,\quad h1=\operatorname{ReLU}(z1),\quad \hat y=w2h1+b2,\quad L=\frac12(\hat y-y)^2$$

손실은 **한 샘플의 오차 제곱의 절반**입니다. 여러 샘플을 평균한 손실은 아닙니다. `forward_and_grad(theta, x, y)`는 주어진 파라미터로 **출력, 손실, 기울기**를 구해 돌려주는 제공 함수입니다. 파라미터를 수정하지는 않습니다.

`pred, loss, g = forward_and_grad(...)`처럼 쓰면 반환되는 세 결과를 왼쪽 이름에 각각 저장합니다. 함수의 코드를 다시 작성하는 것은 과제가 아닙니다.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("concrete_data.csv")
df.columns = df.columns.str.strip()
x = df["cement"].to_numpy()[0] / 100.0
y = df["concrete_compressive_strength"].to_numpy()[0]
theta0 = np.array([0.5, 0.1, 20.0, 20.0])

print("전체 데이터 크기:", df.shape)
print("첫 샘플의 계산용 입력 / 실제 정답:", x, y)
print("파라미터 순서: [w1, b1, w2, b2]")
print("초기 파라미터:", theta0)


전체 데이터 크기: (1030, 9)
첫 샘플의 계산용 입력 / 실제 정답: 5.4 79.99
파라미터 순서: [w1, b1, w2, b2]
초기 파라미터: [ 0.5  0.1 20.  20. ]


In [5]:
# 2장 1강의 순전파·역전파를 묶은 제공 함수입니다.
def forward_and_grad(theta, x_value, y_value):
    w1, b1, w2, b2 = theta
    z1 = w1 * x_value + b1
    h1 = np.maximum(0.0, z1)
    prediction = w2 * h1 + b2
    error = prediction - y_value
    loss = 0.5 * error ** 2

    # z1 = 0에서는 코드상 미분값을 0으로 두는 규칙을 사용합니다.
    # 이번 문제들의 계산 지점은 모두 z1 > 0입니다.
    relu_slope = 1.0 if z1 > 0 else 0.0
    grad_z1 = error * w2 * relu_slope
    gradients = np.array([
        grad_z1 * x_value,  # w1 기울기
        grad_z1,            # b1 기울기
        error * h1,         # w2 기울기
        error               # b2 기울기
    ])
    return prediction, loss, gradients

pred0, loss0, g0 = forward_and_grad(theta0, x, y)



print("초기 출력 / 손실:", round(pred0, 8), round(loss0, 8))
print("초기 기울기:", np.round(g0, 8))


초기 출력 / 손실: 76.0 7.96005
초기 기울기: [-430.92   -79.8    -11.172   -3.99 ]


---

## 실습 1. SGD와 Learning Rate 이해하기


### 문제 1-1. 같은 기울기에서 학습률만 바꾸기

#### 문제 설명

역전파는 각 파라미터의 기울기를 구하고, Optimizer는 그 기울기로 실제 파라미터 값을 바꿉니다. 기본 SGD는 다음 식을 사용합니다.

$$\theta_{new}=\theta_{old}-lr\times g$$

SGD는 보통 샘플 하나 또는 작은 묶음으로 기울기를 계산해 반복 업데이트합니다. **이번에는 샘플 선택에 따른 차이를 없애기 위해 첫 샘플 하나를 고정하고, 기본 업데이트 식만 관찰**합니다. 무작위로 전체 데이터를 순회하는 학습 루프는 아닙니다.

준비 코드의 theta0, g0, x, y를 사용해 학습률 **0.00001, 0.0001, 0.001**을 비교합니다. 세 경우 모두 **동일한 theta0에서 독립적으로 한 번** 출발합니다.

#### 수식을 말로 읽기

`lr*g`는 기존 파라미터에서 뺄 양입니다. 기울기가 음수이면 음수를 빼므로 값이 증가합니다. 학습률이 10배 커지면 **같은 기울기에서** 이 수정량도 10배 커집니다. 실제 손실 감소량까지 10배가 된다는 뜻은 아닙니다.

#### 읽기 도움말 — ‘확률적’이라는 이름의 의미

SGD는 Stochastic Gradient Descent, 우리말로 확률적 경사하강법입니다. 보통 전체 데이터 대신 선택한 샘플 하나 또는 작은 묶음에서 구한 기울기로 업데이트합니다. 무작위로 가중치를 아무 방향이나 바꾼다는 뜻은 아닙니다. 선택한 데이터에서 구한 기울기를 사용해 정해진 식으로 수정합니다.

이번에는 샘플 선택의 차이를 없애고 업데이트 식만 보기 위해 첫 샘플을 고정합니다. 세 학습률도 서로 이어서 적용하지 않고 같은 초기값에서 각각 한 번 적용합니다. 모델의 가중치·편향은 학습으로 바뀌는 파라미터이고, 학습률은 이번 실습에서 우리가 정해 주는 설정값입니다.

#### 요구사항

1. theta0와 g0를 출력하고 두 배열의 순서를 확인하세요.
2. 세 학습률로 `theta_small`, `theta_mid`, `theta_large`를 각각 계산하세요.
3. 각 새 파라미터를 제공 함수에 넣어 수정 후 출력과 손실을 구하세요.
4. 학습률별 파라미터, 출력, 손실을 출력하세요.
5. w1에 대한 세 수정량과 새 w1 값을 손계산하세요.

#### 작성 도움말

`theta_small = theta0 - 0.00001 * g0` 형식을 사용합니다. 나머지도 theta0에서 시작해야 합니다. `pred_small, loss_small, g_small = forward_and_grad(theta_small, x, y)`로 수정 후 결과를 받습니다. 이번 문제에서는 반환된 새 기울기를 추가 업데이트에 사용하지 않습니다.

#### 해석 질문

**Q1.** Optimizer와 역전파는 각각 무엇을 하나요?

**Q2.** w1의 기울기 -430.92에 세 학습률을 적용하면 새 w1은 각각 얼마인가요?

**Q3.** 세 학습률의 수정 후 출력과 손실은 각각 얼마인가요? 이 예시에서 가장 작은 손실은 어느 경우인가요?

**Q4.** 학습률이 크면 항상 더 빨리 좋아지나요? 세 경우를 같은 초기값에서 비교하는 이유는 무엇인가요?

#### 제출 결과

- 세 경우의 파라미터·출력·손실
- w1 손계산 및 Q1~Q4 답변


In [4]:
# 준비 코드의 theta0, g0, x, y를 사용합니다.
# 세 실험은 모두 theta0에서 시작하세요.

# 1. theta_small = theta0 - 0.00001 * g0
# 2. theta_mid: 학습률 0.0001
# 3. theta_large: 학습률 0.001
# 각 새 파라미터로 forward_and_grad를 호출하고 출력하세요.
theta_small = theta0 - 0.00001 * g0
theta_mid = theta0 - 0.0001 * g0
theta_large = theta0 - 0.001 * g0

pred_small, loss_small, g_small = forward_and_grad(theta_small, x, y)
pred_mid, loss_mid, g_mid = forward_and_grad(theta_mid, x, y)
pred_large, loss_large, g_large =forward_and_grad(theta_large, x, y) 

print('초기 출력 / 손실:', round(pred0, 8), round(loss0, 8))
print('lr=0.00001 파라미터:', np.round(theta_small,8))
print('lr=0.00001 출력/손실:',np.round(theta_mid,8))
print('lr=0.0001 파라미터:',np.round(theta_mid,8))
print('lr=0.0001 출력/손실:',round(pred_mid, 8))
print('lr=0.001 파라미터: ', np.round(theta_large,8))
print('lr=0.001 출력/손실:', round(pred_large, 8), round(loss_large, 8))


초기 출력 / 손실: 76.0 7.96005
lr=0.00001 파라미터: [ 0.5043092   0.100798   20.00011172 20.0000399 ]
lr=0.00001 출력/손실: [ 0.543092   0.10798   20.0011172 20.000399 ]
lr=0.0001 파라미터: [ 0.543092   0.10798   20.0011172 20.000399 ]
lr=0.0001 출력/손실: 80.81733204
lr=0.001 파라미터:  [ 0.93092   0.1798   20.011172 20.00399 ]
lr=0.001 출력/손실: 124.19752001 977.15241281


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. Optimizer와 역전파는 각각 무엇을 하나요?**

- 답변: 역전파는 현재 파라미터에서의 손실 변화율인 기울기를 각 파라미터에 대해 구한다.
- Optimizer는 그 기울기와 학습률, 정해진 규칙에 따라 새 파라미터를 만든다.
- forward_and_grad()는 순전파, 손실, 역전파를 수행하여 결과를 돌려주지만 theta를 수정하지는 않음
- theta_new = theta_old - lr * g 같은 줄에서 새 값이 계산됨

**Q2. w1의 기울기 -430.92에 세 학습률을 적용하면 새 w1은 각각 얼마인가요?**

- 답변: 기준 w1은 0.5이다. lr=0.00001이면 0.5 - 0.00001 x (-430.92) = 0.5 + 0.0043092 = 0.5043092
- lr이 0.0001이면 0.543092, lr이 0.001이면 0.93092
- 셋 모두 음수인 lr x g를 빼기 때문에 w1이 증가한다.
- 같은 기울기에서 학습률이 10배씩 커졌기 때문에 증가량도 0.0043092 -> 0.043092 -> 0.43092로 10배씩 상승
- 각 경우는 직전 결과를 반영한 것이 아니라 처음 0.5에서 따로 시작함

**Q3. 세 학습률의 수정 후 출력과 손실은 각각 얼마인가요? 이 예시에서 가장 작은 손실은 어느 경우인가요?**

- 답변: lr = 0.00001에서는 출력 약 76.48170900, 손실 약 6.15405285
- lr = 0.0001에서는 출력 약 80.81733204, 손실 약 0.34223916
- lr = 0.001에서는 출력 약 124.19752001, 손실 약 977.15241281
- 가장 손실이 작은 경우는 0.001 <- 다른 학습률도 봐야하겠지만 현재로서는 이 학스뷸ㄹ을 선택하는 것이 합리적

**Q4. 학습률이 크면 항상 더 빨리 좋아지나요? 세 경우를 같은 초기값에서 비교하는 이유는 무엇인가요?**

- 답변: 항상 그렇지 않음, 큰 학습률은 같은 기울기에서 수정 폭을 키우지만 너무 커지는 경우 정답에서 멀어질 수 있음
- 이번 0.001에서는 손실이 초기 7.96보다 훨씬 커졌기 때문에 같은 초기값, 입력, 기울기 등을 유지해야 학습률만 바꾼 효과를 비교할 수 있다.


---

## 실습 2. Adam의 기록과 SGD와의 차이 확인하기


### 문제 2-1. Adam의 첫 번째 업데이트

#### 문제 설명

SGD 실험과 독립적으로 theta0에서 다시 시작합니다. Adam은 현재 기울기를 그대로 학습률과 곱하는 대신, 기울기와 제곱 기울기의 기록을 이용합니다. 기본 학습률은 **lr=0.0001**로 정합니다.

#### 제공된 Adam 계산식

초기 기록은 `m=0`, `v=0`이고, 첫 순번은 t=1입니다. 이번 설정은 **beta1=0.9, beta2=0.999, eps=1e-8**로 고정합니다. lr은 별도로 설정합니다.

$$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$$
$$v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$$
$$\hat m_t=\frac{m_t}{1-\beta_1^t},\qquad \hat v_t=\frac{v_t}{1-\beta_2^t}$$
$$\theta_t=\theta_{t-1}-lr\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}$$

| 기호 / 코드 | 말로 읽기 |
|---|---|
| g | 현재 파라미터에서 구한 기울기 |
| m | 이전 기울기 기록 90%와 현재 기울기 10%를 합침 |
| v | 이전 제곱 기울기 기록 99.9%와 현재 제곱값 0.1%를 합침 |
| m_hat, v_hat | 0에서 시작한 기록의 초기값 영향을 보정한 값 |
| beta1 ** t | beta1을 t번 곱함. beta1×t와 다름 |
| adam_step | 파라미터에서 빼 줄 양: lr×m_hat/(sqrt(v_hat)+eps) |

배열의 제곱·곱셈·나눗셈은 **같은 위치의 원소별 계산**입니다. 행렬곱 `@`를 쓰지 않습니다. eps는 제곱근을 구한 **뒤에 더합니다**.

Adam의 초기값 보정은 m과 v가 0에서 시작하여 초기 평균이 0 쪽으로 치우치는 영향을 줄이는 과정입니다. 모델 편향 b1·b2만 수정하는 과정이 아닙니다. 증명이나 암기는 요구하지 않고, 이 식을 순서대로 적용합니다.

#### 읽기 도움말 — Adam이 ‘기록한다’는 뜻

Adam은 Adaptive Moment Estimation이라는 이름에서 나온 방식입니다. 지금은 영문 용어를 외우기보다 **기울기를 요약한 m과 기울기 제곱을 요약한 v를 다음 단계에도 이어 쓴다**는 점에 집중하세요.

m과 v는 모든 과거 기울기를 긴 목록으로 저장하는 것이 아닙니다. 이전까지의 정보를 담은 숫자에 현재 기울기 정보를 섞어 요약값을 갱신합니다. 파라미터가 네 개이므로 m도 네 값, v도 네 값을 가집니다. 각 위치가 같은 파라미터에 대응합니다.

`adam_step`은 기존 값에서 **뺄 부호 있는 수**입니다. 실제 변화량인 ‘새 값 − 기존 값’은 그 반대 부호입니다. adam_step이 음수라면 그 수를 빼면서 파라미터는 증가합니다.

#### 요구사항

1. 제공 코드의 초기 기록 m0, v0와 순번 t=1을 사용하세요.
2. 현재 기울기 g0로 m1, v1을 계산하세요.
3. m_hat1, v_hat1을 계산하세요.
4. `adam_step1 = lr * m_hat1 / (np.sqrt(v_hat1) + eps)`를 계산하세요.
5. `theta_adam1 = theta0 - adam_step1`로 새 파라미터를 구하세요.
6. 새 출력·손실을 구하고 기록·수정량·파라미터와 함께 출력하세요.
7. w1 하나에 대해 m1, v1, 보정값과 새 w1을 손계산하세요. 다른 세 값의 긴 소수 계산은 코드 결과로 확인해도 됩니다.

#### 작성 도움말

`m1 = beta1 * m0 + (1 - beta1) * g0`입니다. v1의 마지막 항에는 g0가 아니라 `g0 ** 2`를 사용합니다. 다음 문제에서 **보정 전 기록 m1과 v1**을 계속 사용하므로 지우거나 다른 값으로 덮어쓰지 마세요.

#### 해석 질문

**Q1.** w1에 해당하는 m1과 v1, m_hat1과 v_hat1은 각각 얼마인가요?

**Q2.** 첫 단계에서 네 파라미터가 각각 약 0.0001 증가한 이유는 무엇인가요? Adam이 언제나 일정한 크기로만 수정한다는 뜻인가요?

**Q3.** m과 v는 무엇을 기록하고, 초기값 보정은 모델의 편향 b1·b2와 어떻게 다른가요?

**Q4.** Adam도 Learning Rate가 필요한가요? eps는 어디에 더하나요?

#### 제출 결과

- m1·v1·보정값·adam_step1·새 파라미터 출력
- 새 출력·손실 및 w1 손계산
- Q1~Q4 답변


In [ ]:
# SGD 결과를 이어 쓰지 않고 theta0에서 시작합니다.
lr = 0.0001
beta1 = 0.9
beta2 = 0.999
eps = 1e-8
t = 1
m0 = np.zeros_like(theta0)
v0 = np.zeros_like(theta0)

# 아래부터 작성하세요.
# m1 → v1 → m_hat1 → v_hat1 → adam_step1 → theta_adam1
# 새 파라미터로 forward_and_grad를 호출해 결과를 출력하세요.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. w1에 해당하는 m1과 v1, m_hat1과 v_hat1은 각각 얼마인가요?**

- 답변: 

**Q2. 첫 단계에서 네 파라미터가 각각 약 0.0001 증가한 이유는 무엇인가요? Adam이 언제나 일정한 크기로만 수정한다는 뜻인가요?**

- 답변: 

**Q3. m과 v는 무엇을 기록하고, 초기값 보정은 모델의 편향 b1·b2와 어떻게 다른가요?**

- 답변: 

**Q4. Adam도 Learning Rate가 필요한가요? eps는 어디에 더하나요?**

- 답변:


### 문제 2-2. 기록을 유지한 두 번째 업데이트와 SGD 비교

#### 문제 설명

문제 2-1을 먼저 실행합니다. **같은 첫 샘플에서 두 번째 업데이트**를 수행합니다. Adam은 theta_adam1에서 출발하고, m1·v1을 유지하며 t=2로 올립니다.

비교용 SGD도 theta0에서 출발해 같은 첫 샘플에 lr=0.0001로 두 번 업데이트합니다. 비교 조건은 다음과 같습니다.

| 항목 | SGD | Adam |
|---|---|---|
| 초기값 | theta0 | theta0 |
| 사용 데이터 | 첫 샘플 하나, 두 번 사용 | 첫 샘플 하나, 두 번 사용 |
| 기본 학습률 | 0.0001 | 0.0001 |
| 업데이트 횟수 | 2회 | 2회 |
| 기울기 | 매번 현재 파라미터에서 재계산 | 매번 현재 파라미터에서 재계산 |
| 추가 기록 | 이번 기본형에는 없음 | m·v·순번 t 유지 |

같은 숫자의 학습률이라도 두 알고리즘에 각각 최적인 설정은 아닙니다. 원리 관찰을 위한 비교이며 성능 우열을 확정하는 실험이 아닙니다.

#### 읽기 도움말 — 무엇을 이어 쓰고 무엇을 다시 구하나요?

같은 학습의 다음 단계에서는 **현재 파라미터와 보정 전 m·v를 이어 사용**합니다. 새 파라미터에서 기울기를 다시 구하고, t를 1에서 2로 올려 보정합니다. 보정한 m_hat·v_hat을 원래 기록 m·v 대신 넣지 않습니다.

데이터를 같은 샘플로 유지해도 파라미터가 바뀌면 출력과 기울기가 달라질 수 있습니다. 따라서 ‘데이터가 같으니 기울기도 그대로 써도 된다’고 생각하지 마세요. SGD 경로와 Adam 경로도 각자의 현재 파라미터에서 계산해야 합니다.

#### 요구사항

1. t=2로 설정하고 g_adam2를 이용해 `m2 = beta1*m1 + (1-beta1)*g_adam2`와 v2를 계산하세요.
2. t=2에 맞춰 m_hat2와 v_hat2를 구하세요.
3. theta_adam1에서 두 번째 수정량을 빼 theta_adam2를 구하세요.
4. Adam의 2회 후 출력·손실을 계산하세요.
5. SGD는 theta0에서 한 번 수정한 theta_sgd1을 만들고, **그 위치에서 기울기를 다시 계산**한 뒤 theta_sgd2를 구하세요.
6. 두 방식의 1회·2회 후 손실과 최종 파라미터를 출력하세요. Adam의 두 번째 기울기·m2·v2도 출력하세요.

#### 작성 도움말

v2는 `beta2*v1 + (1-beta2)*(g_adam2 ** 2)`입니다. m_hat1·v_hat1이 아니라 **m1·v1**을 다음 기록 계산에 사용합니다. 보정 분모는 `1-beta1**2`, `1-beta2**2`입니다.

SGD의 새 기울기는 `pred_sgd1, loss_sgd1, g_sgd2 = forward_and_grad(theta_sgd1, x, y)`로 얻습니다. g0를 두 번 재사용하지 마세요.

#### 해석 질문

**Q1.** 두 번째 Adam 단계에서 m·v를 0으로 다시 만들면 무엇이 달라지나요? t는 얼마인가요?

**Q2.** 왜 두 번째 업데이트의 기울기를 새로 계산해야 하나요? SGD와 Adam이 두 번째에 같은 기울기를 사용하나요?

**Q3.** 두 번 업데이트한 후 SGD와 Adam의 손실은 각각 얼마인가요? 이것만으로 SGD가 언제나 더 좋다고 결론 내릴 수 있나요?

**Q4.** 기본 SGD와 Adam은 현재 기울기와 과거 기록을 어떻게 다르게 사용하나요?

#### 제출 결과

- Adam의 두 번째 기울기·m2·v2
- 두 방식의 1회·2회 손실, 최종 파라미터와 출력
- Q1~Q4 답변


In [ ]:
# 문제 2-1의 theta_adam1, m1, v1, g_adam2를 이어 사용하세요.
t = 2

# 아래부터 작성하세요.
# Adam: 기록 갱신 → t=2 보정 → 두 번째 업데이트
# SGD: theta0에서 1회 → 새 기울기 계산 → 2회
# 같은 첫 샘플, 같은 lr=0.0001을 사용하고 결과를 출력하세요.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 두 번째 Adam 단계에서 m·v를 0으로 다시 만들면 무엇이 달라지나요? t는 얼마인가요?**

- 답변: 

**Q2. 왜 두 번째 업데이트의 기울기를 새로 계산해야 하나요? SGD와 Adam이 두 번째에 같은 기울기를 사용하나요?**

- 답변: 

**Q3. 두 번 업데이트한 후 SGD와 Adam의 손실은 각각 얼마인가요? 이것만으로 SGD가 언제나 더 좋다고 결론 내릴 수 있나요?**

- 답변: 

**Q4. 기본 SGD와 Adam은 현재 기울기와 과거 기록을 어떻게 다르게 사용하나요?**

- 답변:


---

## 과제. 다른 샘플에서 두 Optimizer의 한 단계 복습


### 문제 3-1. 세 번째 샘플에서 SGD와 Adam 각각 한 번 적용

#### 문제 설명

세 번째 샘플의 시멘트량은 **332.5**, 계산용 입력은 **3.325**, 실제 압축강도는 **40.27**입니다. 두 Optimizer 모두 **원래 theta0=[0.5,0.1,20,20]에서 독립적으로 한 번** 시작합니다.

lr=0.0001, beta1=0.9, beta2=0.999, eps=1e-8을 사용합니다. 과제는 새 실험이므로 Adam의 m·v는 0, 순번은 t=1로 시작합니다. 실습에서 얻은 theta_adam2·m2·v2를 이어 쓰지 않습니다.

#### 제공된 계산 순서

1. 제공 함수로 초기 출력·손실·기울기를 구합니다.
2. SGD: `theta_sgd = theta0 - lr*g`.
3. Adam: `m = 0.9*m0 + 0.1*g`, `v = 0.999*v0 + 0.001*g**2`.
4. t=1에 맞춰 m과 v를 보정합니다.
5. `theta_adam = theta0 - lr*m_hat/(np.sqrt(v_hat)+eps)`.
6. 각 새 파라미터에서 출력과 손실을 다시 확인합니다.

#### 읽기 도움말 — 이번 초기화의 이유

Adam 기록을 0으로 만드는 이유는 **독립적인 새 비교 실험**이기 때문입니다. 세 번째 샘플이기 때문만은 아닙니다. 같은 모델의 학습을 계속하면서 다음 샘플이나 묶음으로 넘어가는 경우에는 일반적으로 기록을 이어 사용합니다. 이번 과제는 원래 모델에서 다시 출발하므로 파라미터·기록·순번을 함께 초기 조건으로 맞춥니다.

#### 요구사항

1. 제공 코드의 세 번째 샘플·초기값·설정을 사용하세요.
2. 초기 출력·손실·기울기를 계산해 출력하세요.
3. SGD로 한 번 업데이트하고 새 파라미터·출력·손실을 구하세요.
4. Adam의 초기 기록부터 첫 업데이트를 계산하고 새 파라미터·출력·손실을 구하세요.
5. 두 방식의 수정 전후 손실을 출력하세요.
6. w1 하나에 대한 SGD 수정과 Adam 첫 단계 계산을 적고 Q1~Q5에 답하세요.

#### 작성 도움말

실습 1-1과 2-1의 계산을 그대로 사용합니다. `task_`를 붙인 변수 이름으로 이전 실습과 구분하세요. 이번 초기 기울기는 양수이므로 파라미터가 어느 방향으로 움직이는지 확인하세요.

#### 해석 질문

**Q1.** 세 번째 샘플의 초기 출력·손실·기울기는 무엇인가요?

**Q2.** SGD를 한 번 적용한 새 w1은 얼마이며, 수정 후 손실은 얼마인가요?

**Q3.** Adam 첫 단계에서 w1의 보정된 m·v는 무엇이며, 새 w1과 수정 후 손실은 얼마인가요?

**Q4.** 같은 학습률인데 두 방식의 w1 수정량이 다른 이유는 무엇인가요? 두 방식 모두 증가하나요, 감소하나요?

**Q5.** 이번 과제에서는 왜 Adam 기록을 초기화하나요? 다음 업데이트를 이어 한다면 무엇을 유지하고 무엇을 새로 계산해야 하나요?

#### 제출 결과

- 초기 출력·손실·기울기
- SGD·Adam의 새 파라미터·출력·손실
- w1 손계산 및 Q1~Q5 답변


In [ ]:
task_x = df["cement"].to_numpy()[2] / 100.0
task_y = df["concrete_compressive_strength"].to_numpy()[2]
task_theta0 = np.array([0.5, 0.1, 20.0, 20.0])
task_lr = 0.0001
task_beta1 = 0.9
task_beta2 = 0.999
task_eps = 1e-8
task_t = 1
task_m0 = np.zeros_like(task_theta0)
task_v0 = np.zeros_like(task_theta0)

# 아래부터 작성하세요.
# 초기 기울기를 구한 뒤 두 방식 모두 task_theta0에서 출발합니다.
# SGD 한 번 / Adam 한 번 계산 후 각 출력·손실을 확인하세요.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 세 번째 샘플의 초기 출력·손실·기울기는 무엇인가요?**

- 답변: 

**Q2. SGD를 한 번 적용한 새 w1은 얼마이며, 수정 후 손실은 얼마인가요?**

- 답변: 

**Q3. Adam 첫 단계에서 w1의 보정된 m·v는 무엇이며, 새 w1과 수정 후 손실은 얼마인가요?**

- 답변: 

**Q4. 같은 학습률인데 두 방식의 w1 수정량이 다른 이유는 무엇인가요? 두 방식 모두 증가하나요, 감소하나요?**

- 답변: 

**Q5. 이번 과제에서는 왜 Adam 기록을 초기화하나요? 다음 업데이트를 이어 한다면 무엇을 유지하고 무엇을 새로 계산해야 하나요?**

- 답변:


---

## 실습 마무리

**1. Optimizer가 직접 사용하는 정보와 수행하는 일을 설명하세요.**

- 답변: 

**2. 이번 기본 SGD의 식을 말로 읽어 보세요.**

- 답변: 

**3. Adam에서 m과 v는 각각 무엇인가요?**

- 답변: 

**4. Adam을 쓰면 학습률을 설정하지 않아도 되나요?**

- 답변: 

**5. 다음 업데이트에서 반드시 다시 계산해야 하는 것은 무엇인가요?**

- 답변: 

**6. 이번 결과로 SGD와 Adam의 일반적인 우열을 정할 수 있나요?**

- 답변:
